<a href="https://colab.research.google.com/github/bhattacharya3216/RAG-pipeline/blob/main/RAGfinance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain-community sentence-transformers pandas faiss-gpu-cu12

In [ ]:
import pandas
from google.colab import files
uploaded= files.upload()
data_frame= pandas.read_parquet(list(uploaded.keys())[0])
print(data_frame.columns)

Saving train-00000-of-00001.parquet to train-00000-of-00001 (1).parquet
Index(['_id', 'text', 'reasoning', 'category', 'references', 'answer', 'type'], dtype='object')


In [ ]:
data= data_frame["text"].dropna().tolist()

In [ ]:
from langchain.text_splitter import CharacterTextSplitter
text_splitter=CharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
docs=[]
for t in data:
  docs.extend(text_splitter.split_text(t))
print("Number of chunks: ", len(docs))
print("Sample chunk: ", docs[0][:200])

Number of chunks:  5703
Sample chunk:  Delta in CBOE Data & Access Solutions rev from 2021-23.


In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings

In [ ]:
model= HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

/tmp/ipython-input-2967985017.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  model= HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:86: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
!pip install faiss-cpu
from langchain.vectorstores import FAISS
vectorstore= FAISS.from_texts(docs, model)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 22.5 MB/s eta 0:00:00


In [ ]:
vectorstore

In [ ]:
query= "what are the top 10 ai companies?"
results= vectorstore.similarity_search(query, k=3)
for i, res in enumerate(results):
  print(f"Result {i+1}: {res.page_content[:200]}")

Result 1: Impact of Alphabet Inc.'s AI infrastructure on growth & comp. positioning; GOOGL.
Result 2: In 2023, AMD's acquisitions are driving AI innovation and impacting revenue significantly, TKR: AMD.
Result 3: Impact on revenue & valuation from Microsoft AI integration is significant; MSFT.


In [ ]:
docs = [x.page_content for x in results]

In [ ]:
# pip install accelerate
from transformers import T5Tokenizer, T5ForConditionalGeneration

tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")
model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base", device_map="auto")

input_text = f"based on the information from these chunks: {docs}, answer this query: {query}"
input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to("cuda")

outputs = model.generate(input_ids)
print(tokenizer.decode(outputs[0]))


<pad> Microsoft</s>


In [ ]:
# Step 1: Install Hugging Face libraries
!pip install -q transformers accelerate sentencepiece

# Step 2: Import required classes
from transformers import AutoTokenizer, AutoModelForCausalLM

# Step 3: Load BLOOM model + tokenizer
# You can choose "bigscience/bloom" (176B, very large) or smaller variants like "bigscience/bloom-7b1"
model_name = "bigscience/bloom-7b1"   # smaller variant for Colab

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",   # automatically uses GPU if available
    torch_dtype="auto"   # picks best precision (fp16/bf16)
)

# Step 4: Define a prompt
prompt = f"based on the information from these chunks: {docs}, answer this query: {query}"

# Step 5: Tokenize input
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# Step 6: Generate output
outputs = model.generate(
    **inputs,
    max_length=200,       # limit response length
    temperature=0.7,      # creativity vs determinism
    top_p=0.9             # nucleus sampling
)

# Step 7: Decode and print
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

tokenizer_config.json:   0%|          | 0.00/222 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/739 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


...

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.16G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


based on the information from these chunks: ["Impact of Alphabet Inc.'s AI infrastructure on growth & comp. positioning; GOOGL.", "In 2023, AMD's acquisitions are driving AI innovation and impacting revenue significantly, TKR: AMD.", 'Impact on revenue & valuation from Microsoft AI integration is significant; MSFT.'], answer this query: what are the top 10 ai companies?", "ai companies", "ai companies", "ai companies", "ai companies", "ai companies", "ai companies", "ai companies", "ai companies", "ai companies", "ai companies", "ai companies", "ai companies", "ai companies", "ai companies", "ai companies", "ai companies", "ai companies", "ai companies", "ai companies", "ai companies", "ai companies", "ai companies", "ai companies",
